# Google Threat Intelligence lookup

Put your API key in `notebooks/GoogleThreatIntel/config.json` (copy `config.example.json` if needed). Production jobs use the same client in `htoc.core.gti`.

Run the setup cell once, then paste an indicator (IP, domain, URL, or file hash) in the lookup cell.


In [ ]:
from __future__ import annotations

from typing import Any

import pandas as pd
from IPython.display import Markdown, display

from htoc.core.bootstrap import ensure_htoc_on_path

ensure_htoc_on_path()
from htoc.core.gti import classify_ioc, flatten_gti_report, gui_link, load_gti_api_key, lookup_ioc


def pull_gti(indicator: str) -> dict[str, Any]:
    value = str(indicator).strip()
    kind = classify_ioc(value)
    report = lookup_ioc(value, ioc_type=kind)
    flat = flatten_gti_report(report, indicator=value)
    attrs = ((report.get("data") or {}).get("attributes") or {})
    gti = attrs.get("gti_assessment") or {}
    detections = [
        {
            "engine": engine,
            "category": (row or {}).get("category"),
            "result": (row or {}).get("result"),
        }
        for engine, row in (attrs.get("last_analysis_results") or {}).items()
        if (row or {}).get("category") in {"malicious", "suspicious"}
    ]
    cards = []
    for category, items in (gti.get("gti_description_cards") or {}).items():
        for card in items or []:
            cards.append(
                {
                    "category": category,
                    "influence": card.get("influence"),
                    "title": card.get("title"),
                    "detail": card.get("sub_title"),
                }
            )
    summary = {
        "indicator": value,
        "type": kind,
        "gti_verdict": flat.get("enrich_gti_verdict"),
        "gti_severity": flat.get("enrich_gti_severity"),
        "gti_threat_score": flat.get("enrich_gti_threat_score"),
        "malicious": flat.get("enrich_vtMaliciousCount"),
        "mandiant": flat.get("enrich_gti_mandiant"),
        "threat_actors": flat.get("enrich_gti_threat_actors"),
        "malware_families": flat.get("enrich_gti_malware_families"),
        "gui": gui_link(value, kind),
    }
    return {
        "summary": summary,
        "description": flat.get("enrich_gti_description"),
        "description_cards": cards,
        "detections": detections,
        "flat": flat,
        "report": report,
    }


def show_gti(info: dict[str, Any]) -> None:
    summary = info["summary"]
    display(Markdown(
        f"**{summary['indicator']}** ({summary['type']}) — "
        f"verdict `{summary['gti_verdict']}`, "
        f"severity `{summary['gti_severity']}`, "
        f"score `{summary['gti_threat_score']}`  "
        f"[Open in GTI]({summary['gui']})"
    ))
    if info.get("description"):
        display(Markdown(info["description"]))
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
    if info["description_cards"]:
        display(Markdown("#### GTI explainability cards"))
        display(pd.DataFrame(info["description_cards"]))
    if info["detections"]:
        display(Markdown("#### Malicious / suspicious engines"))
        display(pd.DataFrame(info["detections"]))
    else:
        display(Markdown("_No malicious or suspicious engine detections._"))


key = load_gti_api_key()
print(f"Ready. Key loaded ({len(key)} chars). Paste an indicator in the next cell.")


## Look up an indicator

Set `INDICATOR` to an IP, domain, URL, or MD5 / SHA-1 / SHA-256, then run this cell.


In [ ]:
INDICATOR = "31.14.254.80"

gti_info = pull_gti(INDICATOR)
show_gti(gti_info)
